<div style="background: linear-gradient(135deg, #00B4DB 0%, #0083B0 100%); padding: 40px; border-radius: 12px; border: 1px solid #30363d; text-align: center; color: white;">
  <span style="background: rgba(255,255,255,0.2); border: 1px solid rgba(255,255,255,0.4); color: white; padding: 4px 14px; border-radius: 20px; font-size: 12px; font-weight: 600; text-transform: uppercase;">Kafka Training · Lab 7</span>
  <h1 style="color: #ffffff; font-size: 2.4em; font-weight: bold; margin-top: 15px;">Python Kafka Consumer</h1>
  <p style="color: #e0e0e0; font-size: 1.1em;">Build and run a custom Python consumer to read messages from Kafka.</p>
</div>

---

## 🎯 Overview

In this lab, you will run a custom Python application (`consumer.py`) that subscribes to the `orders` topic and processes the incoming data natively on your machine using the `confluent-kafka` library.

---

## ⚙️ Prerequisites

<div style="background-color: rgba(243, 156, 18, 0.1); border-left: 4px solid #f39c12; padding: 10px 15px; margin: 15px 0; border-radius: 4px;">
  <strong>⚠️ Important:</strong> Ensure any previous multi-broker clusters are stopped. Run <code>docker-compose down</code> in Lab6 before proceeding.
</div>

---

## <span style="color: #00B4DB;">Step 1:</span> Start the Kafka Cluster & Prepare Data


In [ ]:
!docker-compose -f ../../docker-compose.yml up -d

# Create the topic
!docker exec kafka kafka-topics --bootstrap-server localhost:9092 --create --topic orders --partitions 1 --replication-factor 1 --if-not-exists

Let's produce a few dummy orders into the topic so our new Python consumer has something to read!

In [ ]:
!docker exec kafka bash -c "echo -e '9991-{"product":"Tablet"}\n9992-{"product":"Phone"}\n9993-{"product":"Watch"}' | kafka-console-producer --bootstrap-server localhost:9092 --topic orders --property parse.key=true --property key.separator='-'"

---

## <span style="color: #00B4DB;">Step 2:</span> Review `consumer.py`

Open `consumer.py` in your editor. Notice the following:
1. It configures `Consumer` with the broker address.
2. It uses `group.id` to join a consumer group and `auto.offset.reset = earliest` to start reading from the beginning of the topic.
3. It calls `consumer.subscribe(['orders'])` to listen to our topic.
4. It enters a `while` loop calling `consumer.poll()` to fetch batches of records and decodes the bytes back to UTF-8 strings.
5. *Special logic for this lab:* To prevent the notebook cell from hanging forever, the code keeps track of empty polls. If it receives no data for 5 consecutive seconds, it cleanly shuts down!

Let's make sure the required Kafka library is installed.

In [ ]:
%pip install confluent-kafka

---

## <span style="color: #00B4DB;">Step 3:</span> Run the Python Consumer

Now we run our Python program! 
Because of `auto.offset.reset=earliest`, it will read the 3 dummy orders we just produced, print them out, and then wait 5 seconds before gracefully shutting down.

In [ ]:
!python consumer.py

---

## 🧹 Clean Up

Stop the cluster and remove volumes to clean up the disk space.

In [ ]:
!docker-compose -f ../../docker-compose.yml down -v

<div style="background-color: rgba(0, 180, 219, 0.1); border: 1px solid rgba(0, 180, 219, 0.3); padding: 20px; text-align: center; border-radius: 8px; margin-top: 40px;">
  <h3 style="color: #00B4DB; margin-bottom: 10px;">🎉 Lab 7 Complete!</h3>
  <p style="color: #8b949e; margin: 0;">You've successfully run a native Python Kafka Consumer, handled subscriptions, parsed incoming ConsumerRecords, and cleanly shut down a polling loop!</p>
</div>